# Bronze Ingestion — UK Bank Holidays

Ingest official UK bank-holiday reference data from GOV.UK.

This notebook:

1. Loads source configuration.
2. Requests the GOV.UK bank-holiday dataset.
3. Validates the configured region.
4. Lands the original JSON response.
5. Appends the response to the Bronze Delta table.

**Source:** GOV.UK Bank Holidays API  
**Region:** England and Wales  
**Target:** `workspace.urbanpulse_bronze.bank_holidays`

## 1. Initialise project paths

In [0]:
from pathlib import Path
import sys

PROJECT_ROOT = Path.cwd().parents[1]
SRC_PATH = PROJECT_ROOT / "src"

if not SRC_PATH.exists():
    raise FileNotFoundError(
        f"Source directory not found: {SRC_PATH}"
    )

if str(SRC_PATH) not in sys.path:
    sys.path.insert(0, str(SRC_PATH))

print(f"Project root: {PROJECT_ROOT}")

## 2. Import reusable ingestion components

In [0]:
import uuid

from urbanpulse.ingestion.api_clients import ApiClient
from urbanpulse.ingestion.bronze import write_raw_bronze
from urbanpulse.ingestion.landing import land_json
from urbanpulse.utils.config import load_yaml

## 3. Load bank-holiday configuration

In [0]:
CONFIG_PATH = (
    PROJECT_ROOT
    / "conf"
    / "sources.yml"
)

config = load_yaml(
    str(CONFIG_PATH)
)

holiday_config = (
    config["gov_uk"]["bank_holidays"]
)

SOURCE_URL = holiday_config["url"]
REGION = holiday_config["region"]

BRONZE_TABLE = (
    "workspace."
    "urbanpulse_bronze."
    "bank_holidays"
)

LANDING_PATH = (
    "/Volumes/workspace/"
    "urbanpulse_meta/"
    "landing"
)

print(f"Source: {SOURCE_URL}")
print(f"Region: {REGION}")
print(f"Target: {BRONZE_TABLE}")

## 4. Initialise the GOV.UK API client

In [0]:
BASE_URL = "https://www.gov.uk"
ENDPOINT = "/bank-holidays.json"

client = ApiClient(
    base_url=BASE_URL
)

## 5. Request UK bank-holiday data

In [0]:
payload, status_code = client.get(
    endpoint=ENDPOINT
)

request_id = str(
    uuid.uuid4()
)

print(f"HTTP status: {status_code}")
print(f"Request ID: {request_id}")

## 6. Inspect the source response

The response contains separate holiday collections for UK regions.

UrbanPulse uses `england-and-wales`.

In [0]:
print(payload.keys())

In [0]:
payload[REGION]

## 7. Validate the bank-holiday source

The configured region must exist and contain a non-empty `events` collection.

In [0]:
if status_code != 200:
    raise RuntimeError(
        f"GOV.UK returned HTTP "
        f"{status_code}"
    )

if not isinstance(payload, dict):
    raise TypeError(
        "Expected GOV.UK response "
        "to be a dictionary"
    )

if REGION not in payload:
    raise ValueError(
        f"Configured region "
        f"'{REGION}' not found"
    )

region_payload = payload[REGION]

if not isinstance(region_payload, dict):
    raise TypeError(
        "Regional holiday payload "
        "must be a dictionary"
    )

if "events" not in region_payload:
    raise ValueError(
        "Regional payload does not "
        "contain 'events'"
    )

events = region_payload["events"]

if not isinstance(events, list):
    raise TypeError(
        "'events' must be a list"
    )

if not events:
    raise ValueError(
        "No bank-holiday events returned"
    )

print(
    f"Validation passed: "
    f"{len(events)} events returned"
)

## 8. Validate minimum event fields

Every holiday event must contain a title and date.

In [0]:
required_fields = {
    "title",
    "date",
}

invalid_events = []

for event in events:
    missing_fields = (
        required_fields
        - set(event.keys())
    )

    if missing_fields:
        invalid_events.append({
            "event": event,
            "missing_fields": sorted(
                missing_fields
            ),
        })

if invalid_events:
    raise ValueError(
        f"{len(invalid_events)} holiday "
        "events failed validation"
    )

print(
    f"Validated {len(events)} "
    "bank-holiday events."
)

## 9. Inspect a bank-holiday event

In [0]:
events[0]

## 10. Land the original GOV.UK response

Store the complete response rather than only the England and Wales subset so the original source remains recoverable.

In [0]:
landing_file = land_json(
    payload=payload,
    base_path=LANDING_PATH,
    source="gov_uk",
    dataset="bank_holidays",
    request_id=request_id,
)

print(
    f"Raw file landed: "
    f"{landing_file}"
)

## 11. Append the bank-holiday snapshot to Bronze

Each execution stores one complete GOV.UK source snapshot with ingestion metadata.

In [0]:
write_raw_bronze(
    spark=spark,
    payload=payload,
    request_id=request_id,
    source="gov_uk",
    dataset="bank_holidays",
    source_endpoint=ENDPOINT,
    http_status=status_code,
    table_name=BRONZE_TABLE,
)

print(
    f"Bronze bank-holiday ingestion "
    f"completed: {request_id}"
)

## 12. Verify the raw landing file

In [0]:
landing_parent = str(
    Path(landing_file).parent
)

display(
    dbutils.fs.ls(
        landing_parent
    )
)

## 13. Verify the Bronze table

In [0]:
%sql
SELECT
    request_id,
    source,
    dataset,
    source_endpoint,
    ingested_at,
    http_status,
    LENGTH(payload) AS payload_size
FROM workspace.urbanpulse_bronze.bank_holidays
ORDER BY ingested_at DESC;

In [0]:
%sql
SELECT
    COUNT(*) AS snapshots,
    MIN(ingested_at) AS first_ingestion,
    MAX(ingested_at) AS latest_ingestion
FROM workspace.urbanpulse_bronze.bank_holidays;

In [0]:
%sql
SELECT COUNT(*) AS snapshots
FROM workspace.urbanpulse_bronze.bank_holidays;